# Will It Spread? XGBoost Classifier

This notebook implements stage 1 of a two-stage business pipeline:

- Stage 1: predict whether a song will spread to at least one additional country within 60 days
- Stage 2: for songs flagged by stage 1, use the country ranker to predict which countries come next

This version is optimized for **recall subject to a precision floor**.
That matches a business setup where we want to catch as many likely multi-country spreaders as possible, while keeping false positives at a manageable level.


In [ ]:
from pathlib import Path
import json
import os
import tempfile
import warnings

os.environ.setdefault('MPLCONFIGDIR', tempfile.mkdtemp(prefix='matplotlib-'))

import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, brier_score_loss, f1_score, fbeta_score, log_loss, precision_recall_curve, precision_score, recall_score, roc_auc_score
from IPython.display import display

try:
    import xgboost as xgb
except ImportError as exc:
    raise ImportError('Install xgboost first, e.g. `pip install -r requirements.txt`.') from exc

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'datasets').exists() and (candidate / 'requirements.txt').exists():
            return candidate
    raise FileNotFoundError(
        f"Could not locate project root from {start}. Expected a parent containing 'datasets/' and 'requirements.txt'."
    )


ROOT = find_project_root(Path.cwd().resolve())
DATA_DIR = ROOT / 'datasets' / 'v3_features'
MODEL_DIR = ROOT / 'artifacts' / 'models' / 'xgboost_will_spread_classifier'
EVAL_DIR = ROOT / 'artifacts' / 'evaluations' / 'xgboost_will_spread_classifier'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'train.parquet'
VAL_PATH = DATA_DIR / 'val.parquet'
TEST_PATH = DATA_DIR / 'test.parquet'

assert TRAIN_PATH.exists(), f'Missing training split: {TRAIN_PATH}'
assert VAL_PATH.exists(), f'Missing validation split: {VAL_PATH}'
assert TEST_PATH.exists(), f'Missing test split: {TEST_PATH}'

RANDOM_STATE = 42
RUN_FULL_SPLITS = True
DEBUG_MAX_TRAIN_TRACKS = None
DEBUG_MAX_VAL_TRACKS = None
DEBUG_MAX_TEST_TRACKS = None

TRAIN_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TRAIN_TRACKS
VAL_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_VAL_TRACKS
TEST_MAX_TRACKS = None if RUN_FULL_SPLITS else DEBUG_MAX_TEST_TRACKS
RUN_MODE = 'full' if RUN_FULL_SPLITS else 'debug_sampled'

PRIMARY_PRECISION_FLOOR = 0.25
BUSINESS_PRECISION_FLOORS = [0.20, 0.25]
OPTIMIZE_BETA = 2.0

print({
    'run_mode': RUN_MODE,
    'train_max_tracks': TRAIN_MAX_TRACKS,
    'val_max_tracks': VAL_MAX_TRACKS,
    'test_max_tracks': TEST_MAX_TRACKS,
    'primary_precision_floor': PRIMARY_PRECISION_FLOOR,
    'business_precision_floors': BUSINESS_PRECISION_FLOORS,
    'optimize_beta': OPTIMIZE_BETA,
})

In [ ]:
con = duckdb.connect()

TARGET_SPECIFIC_COLS = [
    'artist_prior_success_in_target',
    'target_population',
    'target_avg_daily_streams',
    'target_new_entry_rate_30d',
    'target_continent_africa',
    'target_continent_asia',
    'target_continent_europe',
    'target_continent_north_america',
    'target_continent_oceania',
    'target_continent_south_america',
    'cultural_dist_min',
    'cultural_dist_missing',
    'same_language_flag',
    'same_continent_flag',
    'neighbor_entered_count',
]
EXCLUDE_FROM_TRACK_LEVEL = {'target_country', 'did_enter_within_60d', 'days_to_entry'}


def load_track_level_split(path: Path, max_tracks: int | None = None) -> pd.DataFrame:
    parquet_path = path.as_posix()
    schema_df = con.execute(f"SELECT * FROM read_parquet('{parquet_path}') LIMIT 0").fetchdf()
    raw_cols = list(schema_df.columns)
    constant_cols = [c for c in raw_cols if c not in EXCLUDE_FROM_TRACK_LEVEL and c not in TARGET_SPECIFIC_COLS]
    rank_cols = [c for c in raw_cols if c.startswith('rank_')]

    source_expr = f"read_parquet('{parquet_path}')"
    if max_tracks is None:
        source_query = source_expr
    else:
        source_query = f'''
            (
                WITH sampled_tracks AS (
                    SELECT track_id
                    FROM {source_expr}
                    GROUP BY track_id
                    ORDER BY hash(track_id)
                    LIMIT {int(max_tracks)}
                )
                SELECT d.*
                FROM {source_expr} d
                JOIN sampled_tracks st USING (track_id)
            )
        '''

    select_parts = [
        'track_id',
        'MAX(CAST(did_enter_within_60d AS INTEGER)) AS will_spread',
        'MIN(CASE WHEN did_enter_within_60d = 1 THEN days_to_entry END) AS min_days_to_spread',
        'COUNT(*) AS candidate_count',
    ]
    select_parts.extend([f'ANY_VALUE({col}) AS {col}' for col in constant_cols if col != 'track_id'])
    for col in TARGET_SPECIFIC_COLS:
        select_parts.append(f'AVG({col}) AS {col}_mean')
        select_parts.append(f'MAX({col}) AS {col}_max')

    query = f'''
        SELECT
            {', '.join(select_parts)}
        FROM {source_query}
        GROUP BY track_id
        ORDER BY track_id
    '''
    df = con.execute(query).fetchdf()
    df['observation_time'] = pd.to_datetime(df['observation_time'])
    df['origin_country_count_at_obs'] = (df[rank_cols].fillna(0) > 0).sum(axis=1)
    return df


def make_feature_matrix(df: pd.DataFrame, feature_cols: list[str], fill_values: pd.Series) -> pd.DataFrame:
    return df[feature_cols].copy().fillna(fill_values)


def build_threshold_table(y_true: pd.Series, y_prob: np.ndarray, beta: float = OPTIMIZE_BETA) -> pd.DataFrame:
    precision, recall, thresholds = precision_recall_curve(y_true, y_prob)
    rows = []
    beta_sq = beta ** 2
    for p, r, t in zip(precision[:-1], recall[:-1], thresholds):
        denom = p + r
        f1 = 0.0 if denom == 0 else 2 * p * r / denom
        fbeta = 0.0 if (beta_sq * p + r) == 0 else (1 + beta_sq) * p * r / (beta_sq * p + r)
        predicted_positive_rate = float((y_prob >= t).mean())
        rows.append({
            'threshold': float(t),
            'precision': float(p),
            'recall': float(r),
            'f1': float(f1),
            f'f{beta}': float(fbeta),
            'predicted_positive_rate': predicted_positive_rate,
            'flagged_per_1000_tracks': predicted_positive_rate * 1000.0,
        })
    threshold_df = pd.DataFrame(rows).sort_values(['threshold']).reset_index(drop=True)
    return threshold_df


def choose_recall_threshold(y_true: pd.Series, y_prob: np.ndarray, precision_floor: float) -> tuple[float, pd.DataFrame, str]:
    threshold_df = build_threshold_table(y_true, y_prob)
    eligible = threshold_df[threshold_df['precision'] >= precision_floor].copy()
    if not eligible.empty:
        selected = eligible.sort_values(['recall', 'precision', 'threshold'], ascending=[False, False, True]).iloc[0]
        reason = f'highest recall with precision >= {precision_floor:.2f}'
    else:
        score_col = f'f{OPTIMIZE_BETA}'
        selected = threshold_df.sort_values([score_col, 'recall', 'precision'], ascending=[False, False, False]).iloc[0]
        reason = f'fallback to best {score_col} because no threshold met precision floor {precision_floor:.2f}'
    return float(selected['threshold']), threshold_df, reason


def binary_summary(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    summary = {
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'average_precision': float(average_precision_score(y_true, y_prob)),
        'log_loss': float(log_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6))),
        'brier_score': float(brier_score_loss(y_true, np.clip(y_prob, 1e-6, 1 - 1e-6))),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
        f'f{OPTIMIZE_BETA}': float(fbeta_score(y_true, y_pred, beta=OPTIMIZE_BETA, zero_division=0)),
        'positive_rate': float(np.mean(y_true)),
        'predicted_positive_rate': float(np.mean(y_pred)),
        'flagged_per_1000_tracks': float(np.mean(y_pred) * 1000.0),
    }
    return summary


def subgroup_summary(scored_df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    rows = []
    groups = {
        'one_origin': scored_df['origin_country_count_at_obs'] == 1,
        'multi_origin': scored_df['origin_country_count_at_obs'] > 1,
    }
    for label, mask in groups.items():
        subset = scored_df.loc[mask].copy()
        if subset.empty:
            continue
        metrics = binary_summary(subset['will_spread'].astype(int), subset['score'].to_numpy(), threshold)
        rows.append({
            'group': label,
            'tracks': int(len(subset)),
            'positive_tracks': int(subset['will_spread'].sum()),
            **metrics,
        })
    return pd.DataFrame(rows)


def feature_category(feature: str) -> str:
    if feature.startswith('rank_') or feature in {'candidate_count', 'origin_country_count_at_obs', 'track_in_viral50_at_obs'}:
        return 'current_footprint'
    if feature.startswith('artist_prior_') or feature in {'artist_country_ratio', 'multi_artist_flag'}:
        return 'artist_history'
    if feature.startswith('target_') or feature in {'same_language_flag_mean', 'same_language_flag_max', 'same_continent_flag_mean', 'same_continent_flag_max', 'neighbor_entered_count_mean', 'neighbor_entered_count_max', 'cultural_dist_min_mean', 'cultural_dist_min_max', 'cultural_dist_missing_mean', 'cultural_dist_missing_max'}:
        return 'target_and_relationship_aggregates'
    if feature.startswith('af_') or feature in {'duration_ms', 'explicit', 'days_since_release', 'is_friday_release', 'observation_month', 'observation_year'}:
        return 'audio_track_metadata'
    return 'other'


def build_business_threshold_summary(split_name: str, y_true: pd.Series, y_prob: np.ndarray, floors: list[float]) -> tuple[pd.DataFrame, dict, pd.DataFrame]:
    rows = []
    threshold_tables = []
    threshold_map = {}
    for floor in floors:
        threshold, threshold_df, reason = choose_recall_threshold(y_true, y_prob, precision_floor=floor)
        metrics = binary_summary(y_true, y_prob, threshold)
        row = {
            'split': split_name,
            'precision_floor': float(floor),
            'threshold': float(threshold),
            'selection_reason': reason,
            **metrics,
        }
        rows.append(row)
        threshold_map[floor] = threshold
        threshold_df = threshold_df.copy()
        threshold_df['split'] = split_name
        threshold_df['precision_floor'] = float(floor)
        threshold_tables.append(threshold_df)
    return pd.DataFrame(rows), threshold_map, pd.concat(threshold_tables, ignore_index=True)


In [ ]:
train_df = load_track_level_split(TRAIN_PATH, TRAIN_MAX_TRACKS)
val_df = load_track_level_split(VAL_PATH, VAL_MAX_TRACKS)
test_df = load_track_level_split(TEST_PATH, TEST_MAX_TRACKS)

FEATURE_EXCLUDE = ['track_id', 'observation_time', 'will_spread', 'min_days_to_spread']
feature_cols = [col for col in train_df.columns if col not in FEATURE_EXCLUDE]
fill_values = train_df[feature_cols].median(numeric_only=True)

X_train = make_feature_matrix(train_df, feature_cols, fill_values)
X_val = make_feature_matrix(val_df, feature_cols, fill_values)
X_test = make_feature_matrix(test_df, feature_cols, fill_values)
y_train = train_df['will_spread'].astype(int)
y_val = val_df['will_spread'].astype(int)
y_test = test_df['will_spread'].astype(int)

negatives = int((y_train == 0).sum())
positives = int((y_train == 1).sum())
imbalance_ratio = negatives / max(positives, 1)

print(f"Train tracks: {len(train_df):,} | positives: {int(y_train.sum()):,} ({y_train.mean() * 100:.2f}%)")
print(f"Val tracks:   {len(val_df):,} | positives: {int(y_val.sum()):,} ({y_val.mean() * 100:.2f}%)")
print(f"Test tracks:  {len(test_df):,} | positives: {int(y_test.sum()):,} ({y_test.mean() * 100:.2f}%)")
print(f"Feature count: {len(feature_cols)}")
print(f"imbalance_ratio: {imbalance_ratio:.4f}")


In [ ]:
param_grid = [
    {'scale_pos_weight': 1.0, 'max_depth': 4, 'min_child_weight': 1},
    {'scale_pos_weight': 1.0, 'max_depth': 4, 'min_child_weight': 5},
    {'scale_pos_weight': 1.0, 'max_depth': 6, 'min_child_weight': 1},
    {'scale_pos_weight': 1.0, 'max_depth': 6, 'min_child_weight': 5},
    {'scale_pos_weight': 2.0, 'max_depth': 4, 'min_child_weight': 1},
    {'scale_pos_weight': 2.0, 'max_depth': 4, 'min_child_weight': 5},
    {'scale_pos_weight': 2.0, 'max_depth': 6, 'min_child_weight': 5},
    {'scale_pos_weight': 3.0, 'max_depth': 4, 'min_child_weight': 1},
    {'scale_pos_weight': 3.0, 'max_depth': 4, 'min_child_weight': 5},
    {'scale_pos_weight': 3.0, 'max_depth': 6, 'min_child_weight': 5},
    {'scale_pos_weight': float(imbalance_ratio ** 0.5), 'max_depth': 4, 'min_child_weight': 1},
    {'scale_pos_weight': float(imbalance_ratio ** 0.5), 'max_depth': 6, 'min_child_weight': 5},
]

trial_rows = []
models = {}
threshold_tables = {}
selection_reasons = {}

for trial_idx, params in enumerate(param_grid):
    trial_label = f'trial_{trial_idx:02d}'
    model = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric=['aucpr'],
        tree_method='hist',
        n_estimators=500,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        reg_alpha=0.0,
        max_delta_step=1,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        early_stopping_rounds=30,
        **params,
    )
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    val_scores = model.predict_proba(X_val)[:, 1]
    threshold, threshold_df, reason = choose_recall_threshold(y_val, val_scores, precision_floor=PRIMARY_PRECISION_FLOOR)
    val_metrics = binary_summary(y_val, val_scores, threshold)

    meets_precision_floor = bool(val_metrics['precision'] >= PRIMARY_PRECISION_FLOOR)
    trial_rows.append({
        'trial_label': trial_label,
        'selection_reason': reason,
        'meets_precision_floor': meets_precision_floor,
        'threshold': threshold,
        'best_iteration': None if getattr(model, 'best_iteration', None) is None else int(model.best_iteration + 1),
        **params,
        **val_metrics,
    })
    models[trial_label] = model
    threshold_tables[trial_label] = threshold_df
    selection_reasons[trial_label] = reason

trial_df = pd.DataFrame(trial_rows).sort_values(
    ['meets_precision_floor', 'recall', 'precision', 'average_precision', f'f{OPTIMIZE_BETA}'],
    ascending=[False, False, False, False, False],
).reset_index(drop=True)
best_trial = trial_df.iloc[0].to_dict()
best_trial_label = best_trial['trial_label']
selected_model = models[best_trial_label]
selected_threshold = float(best_trial['threshold'])
selected_threshold_df = threshold_tables[best_trial_label].copy()
selected_threshold_df['trial_label'] = best_trial_label

print('Validation tuning results:')
display(trial_df)
print()
print('Selected trial:')
print(best_trial_label)
print(json.dumps({
    'scale_pos_weight': best_trial['scale_pos_weight'],
    'max_depth': best_trial['max_depth'],
    'min_child_weight': best_trial['min_child_weight'],
    'threshold': best_trial['threshold'],
    'selection_reason': best_trial['selection_reason'],
    'precision': best_trial['precision'],
    'recall': best_trial['recall'],
    f'f{OPTIMIZE_BETA}': best_trial[f'f{OPTIMIZE_BETA}'],
    'average_precision': best_trial['average_precision'],
}, indent=2))


In [ ]:
val_scores = selected_model.predict_proba(X_val)[:, 1]
test_scores = selected_model.predict_proba(X_test)[:, 1]

default_threshold = 0.5
val_default = binary_summary(y_val, val_scores, threshold=default_threshold)
test_default = binary_summary(y_test, test_scores, threshold=default_threshold)

business_val_df, val_threshold_map, business_val_sweep = build_business_threshold_summary('validation', y_val, val_scores, BUSINESS_PRECISION_FLOORS)
threshold_sweep_df = business_val_sweep.copy()
business_rows = []
for floor in BUSINESS_PRECISION_FLOORS:
    threshold = float(val_threshold_map[floor])
    val_metrics = binary_summary(y_val, val_scores, threshold)
    test_metrics = binary_summary(y_test, test_scores, threshold)
    selection_reason = business_val_df.loc[business_val_df['precision_floor'] == float(floor), 'selection_reason'].iloc[0]
    business_rows.append({'split': 'validation', 'precision_floor': float(floor), 'threshold': threshold, 'selection_reason': selection_reason, **val_metrics})
    business_rows.append({'split': 'test', 'precision_floor': float(floor), 'threshold': threshold, 'selection_reason': selection_reason + ' (chosen on validation)', **test_metrics})
business_threshold_df = pd.DataFrame(business_rows)
primary_threshold = float(val_threshold_map[PRIMARY_PRECISION_FLOOR])
val_primary = binary_summary(y_val, val_scores, threshold=primary_threshold)
test_primary = binary_summary(y_test, test_scores, threshold=primary_threshold)

summary_rows = [
    {'split': 'validation', 'policy': 'threshold_0_5', 'threshold': default_threshold, **val_default},
    {'split': 'validation', 'policy': f'precision_floor_{PRIMARY_PRECISION_FLOOR:.2f}', 'threshold': primary_threshold, **val_primary},
    {'split': 'test', 'policy': 'threshold_0_5', 'threshold': default_threshold, **test_default},
    {'split': 'test', 'policy': f'precision_floor_{PRIMARY_PRECISION_FLOOR:.2f}', 'threshold': primary_threshold, **test_primary},
]
summary_df = pd.DataFrame(summary_rows)
print('Overall performance:')
display(summary_df)

print()
print('Business threshold comparison (thresholds chosen on validation only):')
display(business_threshold_df)

val_scored = val_df[['track_id', 'observation_time', 'origin_country_count_at_obs', 'candidate_count', 'will_spread']].copy()
val_scored['score'] = val_scores
test_scored = test_df[['track_id', 'observation_time', 'origin_country_count_at_obs', 'candidate_count', 'will_spread']].copy()
test_scored['score'] = test_scores

for floor, threshold in val_threshold_map.items():
    col = f'pred_will_spread_pfloor_{floor:.2f}'.replace('.', '_')
    val_scored[col] = (val_scored['score'] >= threshold).astype(int)
    test_scored[col] = (test_scored['score'] >= threshold).astype(int)

val_subgroups = subgroup_summary(val_scored, primary_threshold)
val_subgroups['split'] = 'validation'
test_subgroups = subgroup_summary(test_scored, primary_threshold)
test_subgroups['split'] = 'test'
subgroup_df = pd.concat([val_subgroups, test_subgroups], ignore_index=True)
print()
print('One-country vs multi-country spread classification at the primary business threshold:')
display(subgroup_df)

handoff_rows = []
for split_name, scored_df in [('validation', val_scored), ('test', test_scored)]:
    for floor in BUSINESS_PRECISION_FLOORS:
        pred_col = f'pred_will_spread_pfloor_{floor:.2f}'.replace('.', '_')
        flagged = scored_df[pred_col] == 1
        handoff_rows.append({
            'split': split_name,
            'precision_floor': float(floor),
            'tracks_total': int(len(scored_df)),
            'tracks_flagged_to_ranker': int(flagged.sum()),
            'flagged_share': float(flagged.mean()),
            'flagged_per_1000_tracks': float(flagged.mean() * 1000.0),
            'true_spreaders_captured': int(scored_df.loc[flagged, 'will_spread'].sum()),
            'missed_spreaders': int(scored_df.loc[~flagged, 'will_spread'].sum()),
            'false_positives_sent_to_ranker': int(((flagged) & (scored_df['will_spread'] == 0)).sum()),
        })
handoff_df = pd.DataFrame(handoff_rows)
print()
print('Stage-1 handoff to the country ranker:')
display(handoff_df)

print()
print('Top threshold candidates for the selected trial on validation:')
display(selected_threshold_df.sort_values(['precision', 'recall'], ascending=[False, False]).head(15))


In [ ]:
booster = selected_model.get_booster()
importance_gain = booster.get_score(importance_type='gain')
importance_weight = booster.get_score(importance_type='weight')

importance_df = pd.DataFrame({'feature': feature_cols})
importance_df['gain'] = importance_df['feature'].map(importance_gain).fillna(0.0)
importance_df['weight'] = importance_df['feature'].map(importance_weight).fillna(0.0)
importance_df['category'] = importance_df['feature'].map(feature_category)
importance_df = importance_df.sort_values(['gain', 'weight'], ascending=[False, False]).reset_index(drop=True)

print('Top 20 spread-classifier features by gain:')
display(importance_df.head(20))

try:
    fig, axes = plt.subplots(1, 2, figsize=(16, 10))
    plot_gain = importance_df.head(20).sort_values('gain')
    plot_weight = importance_df.sort_values('weight', ascending=False).head(20).sort_values('weight')
    axes[0].barh(plot_gain['feature'], plot_gain['gain'])
    axes[0].set_title('Top 20 Feature Importance (gain)')
    axes[0].set_xlabel('gain')
    axes[1].barh(plot_weight['feature'], plot_weight['weight'])
    axes[1].set_title('Top 20 Feature Importance (weight)')
    axes[1].set_xlabel('weight')
    plt.tight_layout()
    plt.show()
    plt.close(fig)
except Exception as exc:
    print(f'Skipping feature importance plots after runtime error: {exc}')


In [ ]:
training_summary = {
    'config': {
        'run_mode': RUN_MODE,
        'primary_precision_floor': PRIMARY_PRECISION_FLOOR,
        'business_precision_floors': BUSINESS_PRECISION_FLOORS,
        'optimize_beta': OPTIMIZE_BETA,
        'selected_trial': best_trial_label,
        'selected_threshold': primary_threshold,
        'selected_params': {
            'scale_pos_weight': float(best_trial['scale_pos_weight']),
            'max_depth': int(best_trial['max_depth']),
            'min_child_weight': float(best_trial['min_child_weight']),
        },
        'selection_reason': best_trial['selection_reason'],
        'resolved_best_iteration': None if best_trial['best_iteration'] is None else int(best_trial['best_iteration']),
    },
    'data': {
        'train_tracks': int(len(train_df)),
        'val_tracks': int(len(val_df)),
        'test_tracks': int(len(test_df)),
        'train_positive_rate': float(y_train.mean()),
        'val_positive_rate': float(y_val.mean()),
        'test_positive_rate': float(y_test.mean()),
    },
    'feature_cols': feature_cols,
    'fill_values': {col: float(fill_values[col]) for col in feature_cols},
}

evaluation_summary = {
    'validation': {
        'threshold_0_5': val_default,
        f'precision_floor_{PRIMARY_PRECISION_FLOOR:.2f}': val_primary,
    },
    'test': {
        'threshold_0_5': test_default,
        f'precision_floor_{PRIMARY_PRECISION_FLOOR:.2f}': test_primary,
    },
}

model_path = MODEL_DIR / 'xgb_classifier.json'
training_summary_path = MODEL_DIR / 'training_summary.json'
evaluation_summary_path = EVAL_DIR / 'evaluation_summary.json'
summary_path = EVAL_DIR / 'metrics_by_threshold.parquet'
subgroup_path = EVAL_DIR / 'subgroup_metrics.parquet'
importance_path = EVAL_DIR / 'feature_importance.parquet'
val_scores_path = EVAL_DIR / 'val_predictions.parquet'
test_scores_path = EVAL_DIR / 'test_predictions.parquet'
thresholds_path = EVAL_DIR / 'selected_trial_threshold_sweep.parquet'
all_thresholds_path = EVAL_DIR / 'business_threshold_sweep.parquet'
tuning_path = EVAL_DIR / 'tuning_results.parquet'
business_summary_path = EVAL_DIR / 'business_threshold_summary.parquet'
handoff_path = EVAL_DIR / 'ranker_handoff_summary.parquet'

selected_model.save_model(model_path)
with open(training_summary_path, 'w', encoding='utf-8') as f:
    json.dump(training_summary, f, indent=2)
with open(evaluation_summary_path, 'w', encoding='utf-8') as f:
    json.dump(evaluation_summary, f, indent=2)

con.register('summary_df', summary_df)
con.register('subgroup_df', subgroup_df)
con.register('importance_df', importance_df)
con.register('val_scored', val_scored)
con.register('test_scored', test_scored)
con.register('selected_threshold_df', selected_threshold_df)
con.register('threshold_sweep_df', threshold_sweep_df)
con.register('trial_df', trial_df)
con.register('business_threshold_df', business_threshold_df)
con.register('handoff_df', handoff_df)

con.execute(f"COPY summary_df TO '{summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY subgroup_df TO '{subgroup_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY importance_df TO '{importance_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY val_scored TO '{val_scores_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY test_scored TO '{test_scores_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY selected_threshold_df TO '{thresholds_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY threshold_sweep_df TO '{all_thresholds_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY trial_df TO '{tuning_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY business_threshold_df TO '{business_summary_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")
con.execute(f"COPY handoff_df TO '{handoff_path.as_posix()}' (FORMAT PARQUET, COMPRESSION 'zstd')")

print(f'Saved spread classifier to: {model_path}')
print(f'Saved training summary to: {training_summary_path}')
print(f'Saved evaluation summary to: {evaluation_summary_path}')
print(f'Saved business threshold summary to: {business_summary_path}')
print(f'Saved ranker handoff summary to: {handoff_path}')
